# GLMsingle beta-version QC — decoding comparison

Aggregates the per-subject CSVs written by `run_beta_qc_decoding.py` and asks a
single question: **does GLMsingle's denoising/ridge pipeline (B→C→D) improve
decodability on this dataset?**

- **A** = ONOFF, **B** = +FITHRF, **C** = +GLMDENOISE, **D** = +ridge (fracridge).
  **A is written but excluded from decoding** — ONOFF pools every event into one
  on/off beta per voxel, so there's no per-trial dimension to decode. The real
  comparison is **B → C → D**.
- Decoding target: **stimulus category** (LinearSVC, accuracy, chance 0.25), same
  standardized decoder / LOGO-CV on every beta version, run on **both** a
  whole-brain mask and a visual-cortex ROI (Harvard-Oxford occipital+fusiform,
  same mask `run_decoding.py` uses). The ROI is the more diagnostic of the two —
  whole-brain dilutes the category signal with tens of thousands of non-visual
  voxels, which can swamp subtler beta-version differences given only 328 trials.
  Category decoding is a **pipeline-validation probe**, not a result of interest;
  the analyses of interest (reward/value) live in `run_qvalue_*.py`.

B→D is usually but **not guaranteed** monotonic; a dip is informative (e.g. GLMdenoise
removing condition-correlated variance), not a bug.

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Directory holding the per-subject sub-*/sub-*_beta_qc_decoding.csv files
# (the --output-dir passed to run_beta_qc_decoding.py). Edit for your filesystem.
QC_DIR = Path("/Users/hugofluhr/phd_local/data/LearningHabits/dev_sample/bids_dataset/derivatives/glmsingle_qc")

# Type A (ONOFF) is never in the data -- run_beta_qc_decoding.py skips it (no
# per-trial betas to decode). Only B/C/D are ever written.
BETA_ORDER = ["B", "C", "D"]
BETA_LABELS = {"B": "B\n+FITHRF", "C": "C\n+GLMdenoise", "D": "D\n+ridge"}

In [2]:
# Load and concatenate all per-subject QC CSVs
csvs = sorted(glob.glob(str(QC_DIR / "sub-*" / "sub-*_beta_qc_decoding.csv")))
assert csvs, f"No beta-QC CSVs found under {QC_DIR}"
df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
df["beta_type"] = pd.Categorical(df["beta_type"], categories=BETA_ORDER, ordered=True)

n_sub = df["subject"].nunique()
print(f"{len(csvs)} subjects, targets: {sorted(df['target'].unique())}")
df.head()

59 subjects, targets: ['category']


,subject,beta_type,target,metric,value,baseline
0,sub-01,B,category,accuracy,0.353659,0.25
1,sub-01,C,category,accuracy,0.347561,0.25
2,sub-01,D,category,accuracy,0.338415,0.25
3,sub-02,B,category,accuracy,0.503049,0.25
4,sub-02,C,category,accuracy,0.533537,0.25


In [ ]:
# Group summary: mean ± sem per beta_type x mask, plus mean gain relative to type-B
# (type A is excluded from decoding — see intro cell — so B is the comparison baseline)
summary = (df.groupby(["mask", "beta_type"], observed=True)["value"]
             .agg(mean="mean", sem=lambda x: x.std(ddof=1) / np.sqrt(x.count()), n="count")
             .reset_index())

for msk in ["wholebrain", "visualcortex"]:
    s = summary[summary["mask"] == msk].set_index("beta_type")
    base = s.loc["B", "mean"] if "B" in s.index else np.nan
    print(f"\n=== {msk} (accuracy) ===")
    for bt in BETA_ORDER:
        if bt in s.index:
            m, se = s.loc[bt, "mean"], s.loc[bt, "sem"]
            print(f"  {bt}: {m:.3f} ± {se:.3f}   (Δ vs B: {m - base:+.3f})")
summary

In [ ]:
# Faint per-subject B→D trajectories + group mean ± sem, one panel per mask,
# chance marked. Visual-cortex (right) is the more diagnostic panel.
x = np.arange(len(BETA_ORDER))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))

for ax, msk in zip(axes, ["wholebrain", "visualcortex"]):
    sub = df[df["mask"] == msk]
    metric = sub["metric"].iloc[0]
    baseline = sub["baseline"].iloc[0]

    # per-subject paired trajectories
    wide = sub.pivot_table(index="subject", columns="beta_type",
                           values="value", observed=True).reindex(columns=BETA_ORDER)
    for _, row in wide.iterrows():
        ax.plot(x, row.values, color="0.75", lw=0.8, alpha=0.6, zorder=1)

    # group mean ± sem
    g = summary[summary["mask"] == msk].set_index("beta_type").reindex(BETA_ORDER)
    ax.errorbar(x, g["mean"].values, yerr=g["sem"].values, color="#c1121f",
                lw=2.2, marker="o", ms=7, capsize=4, zorder=3, label="mean ± sem")

    ax.axhline(baseline, ls="--", color="0.4", lw=1,
               label=f"chance ({baseline:g})", zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels([BETA_LABELS[b] for b in BETA_ORDER])
    ax.set_ylabel(metric)
    ax.set_title(f"{msk} (n={wide.shape[0]})")
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("GLMsingle beta-version decoding QC", fontweight="bold")
fig.tight_layout()
plt.show()